# 챗봇 평가 하네스 합성 점검
DB/LLM을 호출하지 않고 질문 생성 로직과 지표 계산 흐름을 확인합니다. 출력은 저장하지 마세요.

In [ ]:
from pathlib import Path
import csv, json, sys, tempfile
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/card_benefit').exists()), Path.cwd())  # 노트북 위치와 무관하게 저장소 루트를 찾는다
repo = REPO_ROOT
chat_dir = repo / 'src/card_benefit/payment_card/file_pipeline/chat'
sys.path.insert(0, str(chat_dir))
import eval_questions, run_eval
synthetic_csv_dir = Path(tempfile.mkdtemp(prefix='fp-chat-check-'))
def write_csv(name, cols, rows):
    with (synthetic_csv_dir / name).open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(rows)
write_csv('fp_nodes.csv', ['id','node_type','props'], [
 {'id':'c1','node_type':'Card','props':json.dumps({'product_name':'가나다 카드'})},
 {'id':'b1','node_type':'Benefit','props':json.dumps({'name':'커피 할인'})},
 {'id':'l1','node_type':'Limit','props':json.dumps({'period':'월','amount':20000})},
 {'id':'k1','node_type':'Condition','props':json.dumps({'text':'전월 30만원'})},
 {'id':'t1','node_type':'Target','props':json.dumps({'text':'해외 가맹점'})},
 {'id':'x1','node_type':'Target','props':json.dumps({'text':'상품권 구매'})}])
write_csv('fp_edges.csv', ['id','source_id','target_id','edge_type'], [
 {'id':'e1','source_id':'c1','target_id':'b1','edge_type':'OFFERS'}, {'id':'e2','source_id':'b1','target_id':'l1','edge_type':'HAS_LIMIT'},
 {'id':'e3','source_id':'b1','target_id':'k1','edge_type':'HAS_CONDITION'}, {'id':'e4','source_id':'b1','target_id':'t1','edge_type':'APPLIES_TO'}, {'id':'e5','source_id':'b1','target_id':'x1','edge_type':'EXCLUDES'}])
write_csv('fp_evidence.csv', ['node_id','edge_id','doc_sha256','quote_status'], [{'node_id':'b1','edge_id':'','doc_sha256':'a'*64,'quote_status':'located'}])
questions, skipped = eval_questions.generate([synthetic_csv_dir], per_type=2, seed=7)
questions[:2], skipped


In [ ]:
synthetic_records = [{
  'question': {'expected': ['월', '20000'], 'expected_node_names': ['커피 할인']},
  'result': {'answer': '월 한도 20000원', 'sources': [{'title': '문서 12345678'}], 'subgraph': {'nodes': [{'name': '커피 할인'}]}},
  'elapsed_seconds': 0.25
}]
run_eval.metrics(synthetic_records)


## LLM 없는 검색 평가
합성 그래프에서 seed 선택, hop 탐색, 기대값 재현율과 크기를 계산합니다.

In [ ]:
from pathlib import Path
import sys
chat_dir = repo / 'src/card_benefit/payment_card/file_pipeline/chat'
if str(chat_dir) not in sys.path: sys.path.insert(0, str(chat_dir))
import retrieval_eval
synthetic_nodes = [
 {'id':'c','name':'가나다 카드','label':'Card','props':{}},
 {'id':'b','name':'커피 할인','label':'Benefit','props':{}},
 {'id':'l','name':'월 2만원','label':'Limit','props':{'amount':20000}},
 {'id':'d','name':'문서 abcdef12','label':'Document','props':{'summary':'synthetic'}}]
synthetic_edges = [{'source':'c','target':'b','type':'OFFERS'}, {'source':'b','target':'l','type':'HAS_LIMIT'}, {'source':'b','target':'d','type':'SUPPORTED_BY'}]
synthetic_question = {'id':'demo','type':'card_benefits','question':'가나다 카드 혜택','seed_entities':['가나다 카드'],'expected':['커피 할인'],'expected_node_names':['가나다 카드','커피 할인']}
retrieval_eval.evaluate([synthetic_question], synthetic_nodes, synthetic_edges, seed_mode='contains', hops=2)


In [ ]:
# 합성 노드로 조사 제거와 fuzzy 시드 선택을 점검합니다.
from pathlib import Path
import sys
CHAT_DIR = Path.cwd() / "src/card_benefit/payment_card/file_pipeline/chat"
if not CHAT_DIR.exists():
    CHAT_DIR = Path.cwd()
    while CHAT_DIR != CHAT_DIR.parent and not (CHAT_DIR / "seed_matcher.py").exists():
        CHAT_DIR = CHAT_DIR.parent
sys.path.insert(0, str(CHAT_DIR))
from seed_matcher import select_seeds_fuzzy
synthetic_nodes = [
    {"name": "합성 카드 혜택", "label": "Benefit"},
    {"name": "합성 카드", "label": "Card"},
    {"name": "합성 조건", "label": "Condition"},
]
select_seeds_fuzzy(synthetic_nodes, "합성 카드의 혜택은?")
